In [1]:
import pandas as pd
df = pd.read_csv('Bakery sales.csv')
print(df.head())
print(df.shape)
print(df.info())

   Unnamed: 0        date   time  ticket_number               article  \
0           0  2021-01-02  08:38       150040.0              BAGUETTE   
1           1  2021-01-02  08:38       150040.0      PAIN AU CHOCOLAT   
2           4  2021-01-02  09:14       150041.0      PAIN AU CHOCOLAT   
3           5  2021-01-02  09:14       150041.0                  PAIN   
4           8  2021-01-02  09:25       150042.0  TRADITIONAL BAGUETTE   

   Quantity unit_price  
0       1.0     0,90 €  
1       3.0     1,20 €  
2       2.0     1,20 €  
3       1.0     1,15 €  
4       5.0     1,20 €  
(234005, 7)
<class 'pandas.DataFrame'>
RangeIndex: 234005 entries, 0 to 234004
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Unnamed: 0     234005 non-null  int64  
 1   date           234005 non-null  str    
 2   time           234005 non-null  str    
 3   ticket_number  234005 non-null  float64
 4   article        234005 non

In [2]:
df['unit_price'] = (df['unit_price']
    .str.replace('€','')
    .str.replace(',', '.')
    .str.strip()
    .astype(float))
print(df['unit_price'])

0         0.90
1         1.20
2         1.20
3         1.15
4         1.20
          ... 
234000    0.15
234001    1.20
234002    0.15
234003    1.30
234004    1.30
Name: unit_price, Length: 234005, dtype: float64


## Étape 1 : conversion du prix
La colonne unit_price est au format texte (ex. "0,90 €"), donc impossible de faire des calculs.
J'enlève le symbole €, je remplace la virgule par un point, j'enlève les espaces restants, puis je convertis en nombre décimal (float).

In [3]:
print(df['unit_price'].describe())

count    234005.000000
mean          1.662195
std           1.722720
min           0.000000
25%           1.100000
50%           1.200000
75%           1.500000
max          60.000000
Name: unit_price, dtype: float64


In [9]:
print((df['unit_price'] == 0).sum())
print(df[df['unit_price'] == 0].head(10))

32
       Unnamed: 0        date   time  ticket_number             article  \
78            156  2021-01-02  10:51       150079.0   DIVERS PATISSERIE   
10373       23009  2021-02-19  10:13       156355.0  DIVERS BOULANGERIE   
15360       33726  2021-03-04  12:32       159219.0                   .   
15946       34951  2021-03-05  18:18       159538.0       GD FAR BRETON   
19904       43541  2021-03-18  12:59       161853.0                   .   
24959       54650  2021-04-04  09:53       164878.0                   .   
33569       73667  2021-04-27  16:48       170079.0                   .   
34671       76164  2021-05-01  10:17       170776.0   DIVERS PATISSERIE   
39840       87369  2021-05-13  11:15       173793.0   DIVERS CONFISERIE   
61792      135091  2021-07-10  13:25       186662.0                   .   

       Quantity  unit_price  
78          1.0         0.0  
10373       1.0         0.0  
15360       2.0         0.0  
15946       1.0         0.0  
19904       1.0      

In [13]:
df_propre = df[df['unit_price'] >0]
print(df_propre.shape)

(233973, 7)


## Étape 2 : suppression des prix à 0 €
32 lignes ont un prix de 0 € (soit 0,01 % des données). Leurs noms sont peu fiables ("." ou "DIVERS ..."), ce sont probablement des erreurs de saisie. Je les supprime : le chiffre d'affaires ne change pas, mais les données sont plus propres.
Résultat : 233 973 lignes conservées, dans la variable df_propre.

In [16]:
prix_max = df_propre['unit_price'].max()
print(prix_max)
print(df_propre[df_propre['unit_price'] == prix_max])

60.0
        Unnamed: 0        date   time  ticket_number            article  \
201294      439848  2022-07-26  17:07       269498.0  DIVERS PATISSERIE   

        Quantity  unit_price  
201294       1.0        60.0  


Le prix maximum est de 60 € (1 seule ligne, "DIVERS PATISSERIE", le 26/07/2022). C'est extrême mais plausible (pâtisserie sur commande), je la conserve.

In [18]:
print(df_propre['Quantity'].describe())

count    233973.000000
mean          1.538451
std           1.289666
min        -200.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         200.000000
Name: Quantity, dtype: float64


In [25]:
negatifs = df_propre[df_propre['Quantity'] < 0]
print(len(negatifs))
print(negatifs.head(10))

1294
      Unnamed: 0        date   time  ticket_number             article  \
448          898  2021-01-03  10:41       150265.0            BAGUETTE   
1162        2558  2021-01-05  19:00       150738.0       SPECIAL BREAD   
1163        2559  2021-01-05  19:00       150738.0   DIVERS PATISSERIE   
1164        2560  2021-01-05  19:00       150738.0  DIVERS BOULANGERIE   
1467        3269  2021-01-08  11:58       150941.0               COUPE   
1468        3270  2021-01-08  11:58       150941.0          BOULE 400G   
1469        3271  2021-01-08  11:58       150941.0           CROISSANT   
1470        3272  2021-01-08  11:58       150941.0    PAIN AU CHOCOLAT   
1845        4092  2021-01-09  13:23       151163.0    FORMULE SANDWICH   
2043        4524  2021-01-10  11:43       151280.0           VIK BREAD   

      Quantity  unit_price  
448       -1.0        0.90  
1162      -1.0        2.40  
1163      -1.0        6.00  
1164      -1.0        2.00  
1467      -1.0        0.15  
1468  

## Étape 3 : quantités négatives
1 294 lignes ont une quantité négative. Hypothèse : ce sont des retours ou annulations (vrais produits, prix normaux, parfois un ticket entier en négatif). Cela n'est pas confirmé par une colonne du fichier. Je les conserve : si ce sont bien des annulations, elles retranchent la vente d'origine, et leur poids est faible (environ 0,5 % des lignes).

In [30]:
extremes = df_propre[df_propre['Quantity'].abs() > 50]
print(len(extremes))
print(extremes.head(10))

3
        Unnamed: 0        date   time  ticket_number               article  \
50552       110375  2021-06-12  09:58       179931.0           CAFE OU EAU   
50553       110378  2021-06-12  09:58       179932.0           CAFE OU EAU   
115245      250844  2021-10-31  10:42       217821.0  TRADITIONAL BAGUETTE   

        Quantity  unit_price  
50552      200.0         1.0  
50553     -200.0         1.0  
115245      55.0         1.2  


## Étape 4 : quantités extrêmes
3 lignes ont une quantité de plus de 50 en valeur absolue. Deux (+200 et -200 de "CAFE OU EAU", tickets consécutifs, même minute) s'annulent probablement : erreur de saisie corrigée. La 3e (55 baguettes, ~66 €) est plausible. Je conserve les 3 lignes, l'impact sur le chiffre d'affaires est nul ou négligeable.

In [31]:
df_propre['date'] = pd.to_datetime(df_propre['date'])
print(df_propre['date'].dtype)

datetime64[us]


In [33]:
df_propre['ca'] = df_propre['Quantity'] * df_propre['unit_price']
print(df_propre[['Quantity', 'unit_price', 'ca']])
print(df_propre['ca'].sum())

        Quantity  unit_price    ca
0            1.0        0.90  0.90
1            3.0        1.20  3.60
2            2.0        1.20  2.40
3            1.0        1.15  1.15
4            5.0        1.20  6.00
...          ...         ...   ...
234000       1.0        0.15  0.15
234001       1.0        1.20  1.20
234002       2.0        0.15  0.30
234003       1.0        1.30  1.30
234004       1.0        1.30  1.30

[233973 rows x 3 columns]
558640.85


In [34]:
print(df_propre['date'].max())
print(df_propre['date'].min())

2022-09-30 00:00:00
2021-01-02 00:00:00


In [36]:
hebdo = df_propre.set_index('date')['ca'].resample('W').sum()
print(hebdo.head())
print(hebdo.tail())
print(len(hebdo))
print(hebdo.sum())

date
2021-01-03    2002.15
2021-01-10    3612.45
2021-01-17    3305.47
2021-01-24    3208.15
2021-01-31    3130.45
Freq: W-SUN, Name: ca, dtype: float64
date
2022-09-04    5446.70
2022-09-11    5728.80
2022-09-18    5340.80
2022-09-25    4416.30
2022-10-02    3536.05
Freq: W-SUN, Name: ca, dtype: float64
92
558640.8500000001


In [38]:
hebdo_propre = hebdo.loc['2021-01-10':'2022-09-25']
print(len(hebdo_propre))
print(hebdo.sum() - hebdo_propre.sum())

90
5538.199999999953


In [39]:
hebdo_propre = hebdo_propre.round(2)
hebdo_propre.to_csv("ventes_hebdo.csv")
print(pd.read_csv("ventes_hebdo.csv").head())

         date       ca
0  2021-01-10  3612.45
1  2021-01-17  3305.47
2  2021-01-24  3208.15
3  2021-01-31  3130.45
4  2021-02-07  3113.50
